# Conflux (`conflux-state`) — 60-Second Interactive Quickstart

**Deterministic, lock-free CRDT shared state for concurrent AI agents** — zero core dependencies, cryptographic signing (`HMAC-SHA256` / `Ed25519`), crash-recoverable journals, MCP server, and drop-in adapters for **LangGraph**, **CrewAI**, and **AutoGen**.

Run every cell top-to-bottom in Google Colab or Jupyter!

In [ ]:
!pip install -q conflux-state

## 1. Lock-Free Multi-Agent Convergence (3 Concurrent Agents, Zero Locks)
Three autonomous agents mutate shared counters, sets, and registers independently on their own replicas, then exchange journals via a single gossip round. Because every value is a join-semilattice, all nodes converge to the **exact same SHA-256 `state_hash`** regardless of message arrival order.

In [ ]:
from conflux import Cluster, InMemoryTransport, state_hash

cluster = Cluster({"planner", "researcher", "verifier"}, transport=InMemoryTransport())

# Agents write concurrently with zero locks or central coordinator
cluster.node("planner").counter_inc("swarm.tasks_completed", 2)
cluster.node("researcher").counter_inc("swarm.tasks_completed", 3)
cluster.node("verifier").counter_inc("swarm.tasks_completed", 1)

cluster.node("planner").set_add("swarm.findings", "race-condition-in-queue")
cluster.node("researcher").set_add("swarm.findings", "missing-hmac-check")
cluster.node("verifier").register_set("swarm.release_status", "ready")

# One gossip sweep merges all replicas via the lattice join
cluster.gossip_round()

for agent in ("planner", "researcher", "verifier"):
    h = state_hash(cluster.node(agent).replica.state)
    print(f"{agent:10s} | tasks={cluster.read(agent, 'swarm.tasks_completed')} | hash={h[:16]}...")

print("Shared findings:", sorted(cluster.read("planner", "swarm.findings")))

## 2. Cryptographic Trust Boundary (`AgentRegistry` + Networked TCP Mesh)
Equip a `Server` with an `AgentRegistry` to enforce per-agent HMAC/Ed25519 signatures, allowed operations, write quotas, and key namespaces. Rogue or hallucinating agents are rejected before touching shared state.

In [ ]:
from conflux import AgentRegistry, Client, Server

registry = AgentRegistry()
registry.register("trusted-worker", "worker-secret", namespaces=("prod.",), quota=50)
registry.register("sandboxed-agent", "sandbox-secret", namespaces=("sandbox.",), quota=10)

node = Server("alpha", ("127.0.0.1", 0), registry=registry).start()
host, port = node.node_address

try:
    with Client(host, port, agent_id="trusted-worker", secret="worker-secret") as c:
        c.counter_inc("prod.deployments", 1)
        print("Authorized write succeeded -> prod.deployments =", c.read("prod.deployments"))

    with Client(host, port, agent_id="sandboxed-agent", secret="sandbox-secret") as rogue:
        try:
            rogue.register_set("prod.deployments", 9999)  # Out of namespace!
        except ConnectionError as err:
            print("Out-of-namespace write blocked cleanly:", err)
finally:
    node.stop()

## 3. Drop-In Framework Adapters (LangGraph, CrewAI, AutoGen & MCP)
Use `conflux.integrations` directly with LangGraph (`ConfluxCheckpointSaver` + `ConfluxStore`), CrewAI (`ConfluxCrewStorage` + `create_crewai_tools`), AutoGen (`ConfluxAutoGenMemory`), or Claude/Cursor MCP (`ConfluxMCPServer`).

In [ ]:
from conflux.integrations import ConfluxCheckpointSaver, ConfluxCrewStorage, ConfluxStore, FallbackMemoryRecord
from conflux.mcp import ConfluxMCPServer

mesh = Cluster({"node-a", "node-b"}, transport=InMemoryTransport())

# Node A writes via LangGraph Store + CrewAI Memory
lg_store_a = ConfluxStore(mesh.node("node-a"))
lg_store_a.counter_inc(("metrics", "tokens"), "total", by=1250)

crew_mem_a = ConfluxCrewStorage(mesh.node("node-a"))
crew_mem_a.save([FallbackMemoryRecord(id="m1", content="Use exponential backoff on peer dials", scope="/ops")])

# Replicate to Node B and inspect via MCP Server!
mesh.gossip_round()
mcp_b = ConfluxMCPServer(mesh.node("node-b"))
snapshot = mcp_b.call_tool("conflux_snapshot", {})
print("Converged keys on Node B:", snapshot["keys"], "| state_hash:", snapshot["state_hash"][:16] + "...")
print("Recalled CrewAI memory on Node B:", crew_mem_a.get_record("m1").content)